In [7]:
import pandas as pd
import json
from pathlib import Path

def build_seq_id_dict(s, prefix, save_path):
    # 获取唯一项
    uniq = sorted(s.drop_duplicates())
    d = {seq: f"{prefix}_{i}" for i, seq in enumerate(uniq, start=1)}

    save_path = Path(save_path)
    with save_path.open("w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False, indent=2)

    return d

def fasta_to_dict(fasta_path: str):
    d = {}
    header = None

    with open(fasta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                header = line[1:].split()[0]   # 取第一个token做key
                d[header] = ""
            else:
                d[header] += line              # 拼接多行序列
    return d
    
    

In [3]:
H1N1_origin = pd.read_csv('./raw/data4model(H1N1).csv')
H3N2_origin = pd.read_csv('./raw/data4model(H3N2).csv')
selected_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'serumName',
                    'virusName', 'label', 'serumDate', 'serumType', 'virusDate', 'serumIslID', 'virusIslID', 'sheet']
modified_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'serumName',
                    'virusName', 'label', 'serumDate', 'Type', 'virusDate', 'serumIslID', 'virusIslID', 'sheet']
                    
H1N1_data = H1N1_origin[selected_columns]
H1N1_data.columns = modified_columns
H3N2_data = H3N2_origin[selected_columns]
H3N2_data.columns = modified_columns

All_data = pd.concat([H1N1_data, H3N2_data]).reset_index(drop=True)

In [4]:
HA_seqs = pd.concat([All_data["seq_a"], All_data["seq_c"]], ignore_index=True)
NA_seqs = pd.concat([All_data["seq_b"], All_data["seq_d"]], ignore_index=True)

HA_mapping = build_seq_id_dict(HA_seqs, "HA", "./HA_map.json")
NA_mapping = build_seq_id_dict(NA_seqs, "NA", "./NA_map.json")

All_data["seq_id_a"] = All_data["seq_a"].map(HA_mapping)
All_data["seq_id_c"] = All_data["seq_c"].map(HA_mapping)
All_data["seq_id_b"] = All_data["seq_b"].map(NA_mapping)
All_data["seq_id_d"] = All_data["seq_d"].map(NA_mapping)

In [ ]:
fasta_path = "./HA_aligned.fasta"
aligned_HA = fasta_to_dict(fasta_path)
All_data["serumHA"] = All_data["seq_id_a"].map(aligned_HA)
All_data["virusHA"] = All_data["seq_id_c"].map(aligned_HA)

In [10]:
All_data.to_csv('./processed/All.csv', index=False)